## Mining Data from static webpage

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL of the ASIC Miner Value website
url = "https://www.asicminervalue.com/efficiency"

# Make a GET request to fetch the HTML content
response = requests.get(url)

if response.status_code == 200:
    # Parse the HTML content with BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the table containing the miner data
    table = soup.find('table', {'class': 'w-full caption-bottom text-sm'})

    # Extract table headers
    headers = [header.text.strip() for header in table.find_all('th')]

    # Extract table rows
    rows = table.find_all('tr')[1:]  # Skip the header row
    print(len(rows))
    data = []
    for row in rows:
        columns = row.find_all('td')
        row_data = [col.text.strip() for col in columns]
        data.append(row_data)

    # Create a DataFrame
    df = pd.DataFrame(data, columns=headers)

    # Save the data to a CSV file
    df.to_csv('data/asic_miner_ASIC_hardware.csv', index=False)

    print("Data scraped and saved to 'asic_miner_ASIC_hardware.csv'")
else:
    print(f"Failed to fetch the page. Status code: {response.status_code}")


/Users/pj/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


25
Data scraped and saved to 'asic_miner_data.csv'


# Mining Data from Dynamic Webpage

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup
import pandas as pd
import time

# -----------------------------
# Setup Selenium Chrome driver
# -----------------------------
options = Options()

# Run without opening a browser window
options.add_argument("--headless=new")

# Prevent some resource issues on macOS/Linux
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")

# Automatically download correct ChromeDriver version
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# -----------------------------
# Open website
# -----------------------------
url = "https://www.asicminervalue.com/efficiency"

driver.get(url)

# Give page time to load
time.sleep(3)

# -----------------------------
# Scroll until all rows load
# -----------------------------
last_height = driver.execute_script(
    "return document.body.scrollHeight"
)

while True:
    # Scroll to bottom
    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight);"
    )

    # Wait for new content to load
    time.sleep(2)

    # Get new page height
    new_height = driver.execute_script(
        "return document.body.scrollHeight"
    )

    # Stop if no new content appears
    if new_height == last_height:
        break

    last_height = new_height

# -----------------------------
# Parse page source
# -----------------------------
html = driver.page_source

driver.quit()

soup = BeautifulSoup(html, "html.parser")

# Find table
table = soup.find(
    "table",
    {"class": "w-full caption-bottom text-sm"}
)

if table is None:
    print("Table not found.")
    exit()

# -----------------------------
# Extract headers
# -----------------------------
headers = [
    th.get_text(strip=True)
    for th in table.find_all("th")
]

# -----------------------------
# Extract rows
# -----------------------------
data = []

rows = table.find_all("tr")[1:]

print(f"Rows found: {len(rows)}")

for row in rows:
    cols = row.find_all("td")

    row_data = [
        col.get_text(" ", strip=True)
        for col in cols
    ]

    if row_data:
        data.append(row_data)

# -----------------------------
# Create DataFrame
# -----------------------------
df = pd.DataFrame(data, columns=headers)

# -----------------------------
# Save CSV
# -----------------------------
output_file = "data/asic_miner_ASIC_hardware.csv"

df.to_csv(output_file, index=False)

print(f"Saved {len(df)} rows to {output_file}")

Rows found: 246
Saved 246 rows to asic_miner_data.csv


## JSON for bitcoin hashrate

In [5]:
import json
import pandas as pd
from datetime import datetime

# -----------------------------
# Load JSON properly
# -----------------------------
with open('data/hash-rate_30DayAverage_AllYears.json', 'r') as f:
    raw = json.load(f)

# Create initial dataframe preserving original structure
data = pd.DataFrame([raw])

# -----------------------------
# Normalize hashrate
# -----------------------------
hashrate = pd.json_normalize(raw['hash-rate'])

data = pd.concat([data, hashrate], axis=1)

data.rename(columns={
    'x': 'hashrate-unixtimestamp',
    'y': 'hashrate[TH/s]'
}, inplace=True)

# -----------------------------
# Normalize price
# -----------------------------
price = pd.json_normalize(raw['market-price'])

data = pd.concat([data, price], axis=1)

data.rename(columns={
    'x': 'price-unixtimestamp',
    'y': 'price[USD]'
}, inplace=True)

# -----------------------------
# Convert timestamps
# -----------------------------
data['hashrate-datetime'] = pd.to_datetime(
    data['hashrate-unixtimestamp'],
    unit='ms'
)

data['price-datetime'] = pd.to_datetime(
    data['price-unixtimestamp'],
    unit='ms'
)

# -----------------------------
# Save CSV
# -----------------------------
data.to_csv('data/bitcoin-hashrate.csv', index=False)

print(data.head())

     metric1       metric2                                          hash-rate  \
0  hash-rate  market-price  [{'x': 1233532800000, 'y': 4.352962610567901e-...   
1        NaN           NaN                                                NaN   
2        NaN           NaN                                                NaN   
3        NaN           NaN                                                NaN   
4        NaN           NaN                                                NaN   

                                        market-price    type average timespan  \
0  [{'x': 1233532800000, 'y': 0}, {'x': 123387840...  linear     30d      all   
1                                                NaN     NaN     NaN      NaN   
2                                                NaN     NaN     NaN      NaN   
3                                                NaN     NaN     NaN      NaN   
4                                                NaN     NaN     NaN      NaN   

   hashrate-unixtimestamp 

In [3]:
import json
import pandas as pd

# -----------------------------
# Load JSON file
# -----------------------------
with open('data/hash-rate_30DayAverage_AllYears.json', 'r') as f:
    raw = json.load(f)

# -----------------------------
# Extract hashrate data
# -----------------------------
hashrate = pd.DataFrame(raw['hash-rate'])

hashrate.rename(columns={
    'x': 'timestamp',
    'y': 'hashrate[TH/s]'
}, inplace=True)

# -----------------------------
# Extract market price data
# -----------------------------
price = pd.DataFrame(raw['market-price'])

price.rename(columns={
    'x': 'timestamp',
    'y': 'price[USD]'
}, inplace=True)

# -----------------------------
# Merge on timestamp
# -----------------------------
data = pd.merge(
    hashrate,
    price,
    on='timestamp',
    how='outer'
)

# -----------------------------
# Convert timestamps
# -----------------------------
data['datetime'] = pd.to_datetime(
    data['timestamp'],
    unit='ms'
)

# Optional: reorder columns
data = data[
    [
        'timestamp',
        'datetime',
        'hashrate[TH/s]',
        'price[USD]'
    ]
]

# -----------------------------
# Save CSV
# -----------------------------
output_path = 'data/bitcoin-hashrate.csv'

data.to_csv(output_path, index=False)

print(f"Saved {len(data)} rows to {output_path}")

Saved 1623 rows to data/bitcoin-hashrate.csv


In [4]:
data

,timestamp,datetime,hashrate[TH/s],price[USD]
0,1233532800000,2009-02-02,4.352963e-06,0.000000
1,1233878400000,2009-02-06,5.191409e-06,0.000000
2,1234224000000,2009-02-10,5.973517e-06,0.000000
3,1234569600000,2009-02-14,6.079566e-06,0.000000
4,1234915200000,2009-02-18,6.165730e-06,0.000000
...,...,...,...,...
1618,1777852800000,2026-05-04,NaN,74474.883667
1619,1778112000000,2026-05-07,9.617581e+08,NaN
1620,1778198400000,2026-05-08,NaN,76097.049000
1621,1778457600000,2026-05-11,9.678580e+08,NaN
